## Notebook to start a spark instance in an ArcGIS Pro Session.

### THE BELOW HAS TO BE THE FIRST IMPORT !

In [1]:
from spark_esri import spark_start, spark_stop

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [3]:
import os

import arcpy
import geofunctions as S

### Check if the jar exists

In [4]:
spark_jars = os.path.expanduser("~/geofunctions-0.28.jar")
os.path.exists(spark_jars)

True

In [5]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    "spark.jars": spark_jars,
}
spark = spark_start(config=config)

In [6]:
spark.version

'3.5.4-esri-1'

### Register the geo functions with SparkSQL.

In [7]:
S.st_register_functions()

### Sample usage. Create random points in the current map extent and place them on the Map.

In [8]:
project = arcpy.mp.ArcGISProject("CURRENT")

extent = project.activeView.camera.getExtent()
xmin = extent.XMin
ymin = extent.YMin
dx = extent.XMax - extent.XMin
dy = extent.YMax - extent.YMin
extent

XMin (Left),-8927490.721808
XMax (Right),-8910338.313406
YMin (Bottom),2963279.853951
YMax (Top),2976975.693359
name (Projected Coordinate System),WGS_1984_Web_Mercator_Auxiliary_Sphere
factoryCode (WKID),3857
linearUnitName (Linear Unit),Meter
name (Geographic Coordinate System),GCS_WGS_1984
factoryCode (WKID),4326
angularUnitName (Angular Unit),Degree
datumName (Datum),D_WGS_1984


In [9]:
extent.spatialReference.factoryCode 

3857

In [10]:
df = (
    spark
    # Create 1000 points.
    .range(1_000)
    # Get random x values
    .withColumn("x", F.rand() * dx + xmin)
    # Get random y values.
    .withColumn("y", F.rand() * dy + ymin)
    # Create a point geometry from implicit x/y.
    .withColumn("geom", S.st_point())
    # Convert the spark dataframe to a feature class to be displayed in the TOC.
)

df.to_feature_class(
        "MyPoints",
        # sp_ref = -1,  # Use current map SR.
        sp_ref = extent.spatialReference.factoryCode,
    )

In [11]:
df.write.parquet("C:\\temp\\MyPoints\\MyPoints", mode="overwrite")

### Create a spark dataframe from a feature class.

In [ ]:
desc = arcpy.Describe("MyPoints")
fields = [f.name for f in desc.fields]
fields

In [ ]:
df = S.to_spark(desc.name, fields=fields)

In [ ]:
df.printSchema()

In [ ]:
df.count()

In [ ]:
(df
     .select(
        "OBJECTID",
        S.st_astext("Shape"),
    )
     .where(F.col("OBJECTID").between(10,15))
     .show(truncate=False)
)

### Create a Spark SQL table from the dataframe and apply some SQL functions.

In [ ]:
df.createOrReplaceTempView("v")

In [ ]:
spark.sql("""
select OBJECTID,ST_ASTEXT(Shape) as WKT
from v
where OBJECTID between 10 and 15
""").show(truncate=False)